# Types and Python                        <img src="img/logo-mse.png" width="200"  />

The design of Python as a programming language leans towards flexibility and ease of use.
Python **dynamic typing system** entails:
 - not specifying types upfront 
 - validating types during runtime instead of compile time
 
Potential drawbacks/challenges: 
 - runtime errors
 - reduced code readability
 - increased maintenance complexity
 
 
<sub>Credits: some materials adapted from [Javier Román. Say Goodbye to Type Errors: Enforcing Python Types with Decorators.](https://blog.stackademic.com/say-goodbye-to-type-errors-enforcing-python-types-with-decorators-497d51851a10)
    </sub>

## Dynamic typing system

It doesn't mean that there are no types. We can assign a value to a variable:

In [29]:
age = 4

And we can find out at runtime what the type is:

In [30]:
type(age)

int

However, this type is not written in stone for this variable (and also thanks to mutability) we can do this:

In [31]:
age = "four"

In [32]:
type(age)

str

So here we have no enfrocing of typing on the `age` variable, which means that we can't be sure if this operation will be valid:

In [33]:
age + 4

TypeError: can only concatenate str (not "int") to str

## Types in functions

Types could also be handy in functions, for declaring types of parameters and result value.

In this case we do not enforce anything on the parameters of this function.

In [34]:
def compute(a,b):
    return a * b

By looking at the type of the function, we have no information about the types of inputs and outputs.

In [35]:
type(compute)

function

If we look at the result type, we can guess what the result type of the function `compute` could be:

In [36]:
result = compute(3,4)

In [37]:
type(result)

int

However, we can see that there is no checking of parameters, and also the result type can change depending on the inputs.

In [38]:
result = compute('3',4)

In [39]:
type(result)

str

## Type hints

Python introduced a feature called type hints in version 3.5. 
The goal of this is to standardize the annotation of types, which can potentially lead to type checking or at least inform the programmer of how function should be used.

For example, a function can be declared as this:

In [40]:
from datetime import datetime, timedelta, date 

def add_days(date: datetime, days: int) -> datetime:
    return date + timedelta(days=days)

and then we can try to pass parameters that match the input types:

In [41]:
add_days(date(2019,12,4),5)

datetime.date(2019, 12, 9)

However, this does not mean that type chacking is performed. For instance if we pass an `int` instead of `datetime`:

In [42]:
add_days(2019,5)

TypeError: unsupported operand type(s) for +: 'int' and 'datetime.timedelta'

The `TypeError` is not thrown when invoking the function but rather inside the method, when the addition cannot be applied to unsupported types.

## Type hints but not checks

This becomes more evident in the next example. Consider type hinds in this addition function:

In [43]:
def add(a: int, b: int) -> int:
    return a + b

If we ask the user to input the operands for this function:

In [47]:
op1 = input("Enter the first number: ")
op2 = input("Enter the second number: ")
result = add(op1, op2)
result

Enter the first number:  3
Enter the second number:  4


'34'

The result is perhaps not the one intended. We clearly see that types are not checked and the hints are only documenting the expected types.

We can explicitly call it using strings and Python will not complain:

In [48]:
add("lala","land")

'lalaland'

## Enforcing types using hints

One way to enforce types is to use the hints, like in this example from Javier Román.

In [49]:
from typing import get_type_hints
def enforce_types(func):
    def wrapper(*args, **kwargs):
        # Get arguments type hints
        type_hints = get_type_hints(func)
        # Get return type hint
        return_hint = type_hints.pop('return')
        for i, (key, hint) in enumerate(type_hints.items()):
            try:
                value = kwargs.get(key, args[i])
                assert isinstance(value, hint), f"Argument {key} must be of type {hint} but {value=} of type={type(value)} provided"
            except IndexError:
                pass
        result = func(*args, **kwargs)
        assert isinstance(result, return_hint)
        return result
    return wrapper

Using the new decorator `@enforce_types` now the function argument types arechecked before the function is executed:

In [50]:
@enforce_types
def add(a: int, b: int) -> int:
    return a + b

num1 = input("Enter the first number: ")
num2 = input("Enter the second number: ")

result = add(num1, num2)

print(f"The sum of {num1} and {num2} is {result}")

Enter the first number:  3
Enter the second number:  4


AssertionError: Argument a must be of type <class 'int'> but value='3' of type=<class 'str'> provided